# Odor Classification
## https://www.pnas.org/doi/10.1073/pnas.2116576119

In [1]:
import os, sys, time
import pandas as pd
import numpy as np

SRC_CSV = "Data/pnas.2116576119.sd01.csv"

FULL_OUT = "odor_classification_all.csv"
SUBSET_OUT = "odor_classification_175.csv"

SEED = 42
RNG = np.random.default_rng(SEED)


MW_PREF_CAP = 350.0

USE_PUBCHEM_REST = True      
PUBCHEM_TIMEOUT = 4.0
NAME_CACHE_FILE = "OtherFiles/smiles_name_cache.csv"

try:
    import requests
except Exception:
    requests = None
    USE_PUBCHEM_REST = False

def log(msg: str):
    print(msg, flush=True)

def standardize_odor_class(x: str) -> str:
    if not isinstance(x, str):
        return np.nan
    s = x.strip().lower()
    if s in {"odorless", "no odor", "odor free", "no-odor", "odourless"}:
        return "Odorless"
    if s in {"odor", "odorous", "has odor", "odour"}:
        return "Odorous"
    return "Odorous"

def load_name_cache(path: str) -> dict:
    if os.path.exists(path):
        try:
            dfc = pd.read_csv(path)
            if {"SMILES","name"}.issubset(dfc.columns):
                return dict(zip(dfc["SMILES"].astype(str), dfc["name"].astype(str)))
        except Exception:
            pass
    return {}

def save_name_cache(cache: dict, path: str):
    try:
        pd.DataFrame({"SMILES": list(cache.keys()), "name": list(cache.values())}).to_csv(path, index=False)
    except Exception:
        pass

def pubchem_name_from_smiles(smiles: str) -> str:
    if not (USE_PUBCHEM_REST and requests):
        return ""
    try:
        url = (
            "https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/smiles/"
            + requests.utils.quote(smiles, safe="")
            + "/property/IUPACName,Title/JSON"
        )
        r = requests.get(url, timeout=PUBCHEM_TIMEOUT)
        if r.status_code != 200:
            return ""
        data = r.json()
        props = data.get("PropertyTable", {}).get("Properties", [])
        if not props:
            return ""
        rec = props[0]
        # Prefer common Title, else IUPAC
        title = rec.get("Title")
        if title:
            return str(title)
        iupac = rec.get("IUPACName")
        if iupac:
            return str(iupac)
    except Exception:
        return ""
    return ""

def resolve_name(smiles: str, cache: dict) -> str:
    smi = (smiles or "").strip()
    if not smi:
        return ""
    if smi in cache:
        return cache[smi]
    nm = pubchem_name_from_smiles(smi)
    cache[smi] = nm if nm else ""
    return cache[smi]

def build_prompts(smiles: str, name: str) -> tuple[str, str]:
    p1 = f"Does [{smiles}] have a detectable odor to humans? Only respond with Odorous or Odorless. Do not write any comments."
    p2n = name if name else "the compound"
    p2 = f"Does [{p2n}] have a detectable odor to humans? Only respond Odorous or Odorless. Do not write any comments."
    return p1, p2

# ========= Core =========
def main():
    t0 = time.time()
    log(f"Start | PubChem REST={'ON' if USE_PUBCHEM_REST else 'OFF'} (timeout={PUBCHEM_TIMEOUT}s)")

    # Load
    df = pd.read_csv(SRC_CSV)
    for col in ("SMILES","odor.class","MW"):
        if col not in df.columns:
            raise ValueError(f"Missing required column: {col}")

    # Normalize
    df["odor.class.std"] = df["odor.class"].apply(standardize_odor_class)
    df["MW"] = pd.to_numeric(df["MW"], errors="coerce")

    log("Class counts (after standardization):")
    log(df["odor.class.std"].value_counts(dropna=False).to_string())

    # Build full table with names + prompts
    cache = load_name_cache(NAME_CACHE_FILE)
    rows = []
    n = len(df)
    for i, r in df.iterrows():
        if i % 200 == 0:
            log(f"  • progress {i}/{n} (name_cache={len(cache)})")
        smi = str(r["SMILES"]).strip()
        if not smi:
            continue
        cls = r["odor.class.std"]
        if pd.isna(cls):
            continue
        mw = r["MW"]
        name = resolve_name(smi, cache)
        p1, p2 = build_prompts(smi, name)
        answer = "Odorous" if cls == "Odorous" else "Odorless"
        other_info = "" if pd.isna(mw) else f"MW={mw:g}"
        rows.append({
            "compound.name_1": name,
            "SMILES_1": smi,
            "compound.name_2": "",
            "SMILES_2": "",
            "OPTIONS": "Odorous; Odorless",
            "question_category": "odor_classification",
            "prompt.1": p1,
            "prompt.2": p2,
            "answer": answer,
            "other_info": other_info,
        })

    save_name_cache(cache, NAME_CACHE_FILE)

    full = pd.DataFrame(rows)
    full.insert(0, "question_ID", np.arange(1, len(full) + 1, dtype=int))
    # enforce column order
    full = full[[
        "question_ID",
        "compound.name_1","SMILES_1",
        "compound.name_2","SMILES_2",
        "OPTIONS","question_category",
        "prompt.1","prompt.2",
        "answer",
        "other_info"
    ]]

    # Save full
    full.to_csv(FULL_OUT, index=False)
    log(f"Wrote {FULL_OUT} (n={len(full)})")

    # ---------- Build EXACT 175 subset ----------
    # Parse MW numeric back from other_info
    mw_num = pd.to_numeric(full["other_info"].str.replace("MW=","", regex=False), errors="coerce")

    odorous_df = full[full["answer"] == "Odorous"].copy()
    odorless_df_all = full[full["answer"] == "Odorless"].copy()
    odorless_pref = odorless_df_all[mw_num[odorless_df_all.index] <= MW_PREF_CAP]

    need_odorous = 88
    need_odorless = 87

    # Sample Odorous
    if len(odorous_df) < need_odorous:
        raise RuntimeError(f"Not enough Odorous to pick {need_odorous} (have {len(odorous_df)}).")
    pick_odorous = odorous_df.sample(n=need_odorous, random_state=SEED)

    # Sample Odorless
    if len(odorless_pref) >= need_odorless:
        pick_odorless = odorless_pref.sample(n=need_odorless, random_state=SEED)
    else:
        short = need_odorless - len(odorless_pref)
        log(f"Only {len(odorless_pref)} odorless with MW<={MW_PREF_CAP}; topping up {short} by lowest MW odorless overall.")
        # choose remaining from odorless (sorted by MW ascending)
        tmp = odorless_df_all.copy()
        tmp["_mw_num"] = mw_num[tmp.index]
        tmp = tmp.sort_values("_mw_num", ascending=True, na_position="last")
        topup_pool = tmp.loc[~tmp.index.isin(odorless_pref.index)]
        if len(topup_pool) < short:
            raise RuntimeError(f"Not enough odorless to fill {need_odorless}. Available={len(odorless_df_all)}")
        pick_odorless = pd.concat([
            odorless_pref,
            topup_pool.head(short).drop(columns=["_mw_num"], errors="ignore")
        ], ignore_index=False)

        # If odorless_pref was empty, ensure exact count
        if len(pick_odorless) > need_odorless:
            pick_odorless = pick_odorless.sample(n=need_odorless, random_state=SEED)

    # Combine and RANDOMLY SHUFFLE rows (no answer pattern)
    subset = pd.concat([pick_odorous, pick_odorless], ignore_index=True)
    subset = subset.sample(frac=1.0, random_state=SEED).reset_index(drop=True)

    # Reassign question_ID 1..175 for the subset file
    subset["question_ID"] = np.arange(1, len(subset) + 1, dtype=int)

    # Save subset
    subset.to_csv(SUBSET_OUT, index=False)
    log(f"Wrote {SUBSET_OUT} (n={len(subset)}; "
        f"Odorous={(subset['answer']=='Odorous').sum()}, "
        f"Odorless={(subset['answer']=='Odorless').sum()})")

    log(f"Done in {time.time() - t0:.1f}s")

if __name__ == "__main__":
    main()


Start | PubChem REST=ON (timeout=4.0s)
Class counts (after standardization):
odor.class.std
Odorous     1615
Odorless     309
  • progress 0/1924 (name_cache=0)
  • progress 200/1924 (name_cache=200)
  • progress 400/1924 (name_cache=400)
  • progress 600/1924 (name_cache=600)
  • progress 800/1924 (name_cache=800)
  • progress 1000/1924 (name_cache=1000)
  • progress 1200/1924 (name_cache=1200)
  • progress 1400/1924 (name_cache=1400)
  • progress 1600/1924 (name_cache=1600)
  • progress 1800/1924 (name_cache=1800)
Wrote odor_classification_all.csv (n=1924)
Wrote odor_classification_175.csv (n=175; Odorous=88, Odorless=87)
Done in 880.6s
